# 9. Model Saving

In [1]:
from pathlib import Path
import joblib
import pandas as pd

data_dir = Path("../data")
models_path = data_dir / "05_trained_models.pkl"
processed_path = data_dir / "03_processed_data.pkl"
comparison_path = data_dir / "06_model_comparison.csv"

for path in [models_path, processed_path, comparison_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path.resolve()}. "
            "Check that the notebook is inside the project notebooks folder "
            "and that the required data files exist."
        )

trained_models = joblib.load(models_path)
processed_data = joblib.load(processed_path)
comparison = pd.read_csv(comparison_path)

random_forest_model = trained_models["random_forest_model"]
X_train = processed_data["X_train"]

rf_row = comparison.loc[comparison["Model"] == "Random Forest"]
if rf_row.empty:
    raise ValueError("The model comparison CSV does not contain a 'Random Forest' row.")

recall_rf = float(rf_row["Recall"].iloc[0])
roc_auc_rf = float(rf_row["ROC-AUC"].iloc[0])

expected_features = getattr(random_forest_model, "n_features_in_", None)
if expected_features is not None and expected_features != X_train.shape[1]:
    raise ValueError(
        f"Feature mismatch: model expects {expected_features} features, "
        f"but X_train has {X_train.shape[1]}."
    )

print("Training results loaded successfully!")
print("Training data shape:", X_train.shape)
print("Random Forest recall:", round(recall_rf, 4))
print("Random Forest ROC-AUC:", round(roc_auc_rf, 4))

Training results loaded successfully!
Training data shape: (19760, 21)
Random Forest recall: 0.9184
Random Forest ROC-AUC: 0.9134


## Final Model Selection

After comparing Logistic Regression, Decision Tree, and Random Forest,
Random Forest was selected as the final model.

It achieved the highest recall of 91.87%, while maintaining a strong
ROC-AUC of 91.35%.

Recall is emphasized because correctly identifying recovered loans is
an important objective of this project.

In [2]:
final_model = random_forest_model

print("Final model selected: Random Forest")
print(f"Recall: {recall_rf:.4f} ({recall_rf * 100:.2f}%)")
print(f"ROC-AUC: {roc_auc_rf:.4f} ({roc_auc_rf * 100:.2f}%)")

Final model selected: Random Forest
Recall: 0.9184 (91.84%)
ROC-AUC: 0.9134 (91.34%)


In [3]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_data = {
    "model": final_model,
    "feature_columns": X_train.columns.tolist(),
    "model_name": "Random Forest",
    "recall": recall_rf,
    "roc_auc": roc_auc_rf
}

model_path = models_dir / "loan_recovery_model.pkl"
joblib.dump(model_data, model_path)

print(f"Model saved successfully to: {model_path.resolve()}")

Model saved successfully to: C:\Users\manth\Documents\HCL Final Project\models\loan_recovery_model.pkl


In [4]:
saved_model = joblib.load(model_path)

saved_estimator = saved_model["model"]
saved_feature_columns = saved_model["feature_columns"]

print("Saved model loaded successfully!")
print("Model type:", type(saved_estimator).__name__)
print("Number of saved features:", len(saved_feature_columns))
print("Saved model name:", saved_model.get("model_name", "Not recorded"))
print("Saved recall:", round(saved_model.get("recall", float("nan")), 4))
print("Saved ROC-AUC:", round(saved_model.get("roc_auc", float("nan")), 4))

if len(saved_feature_columns) != getattr(saved_estimator, "n_features_in_", len(saved_feature_columns)):
    raise ValueError("Saved feature count does not match the saved model.")

print("Model artifact validation passed!")

Saved model loaded successfully!
Model type: RandomForestClassifier
Number of saved features: 21
Saved model name: Random Forest
Saved recall: 0.9184
Saved ROC-AUC: 0.9134
Model artifact validation passed!
